# Scrapers for MyDramaList

You will follow the instructions in Part 4 of Week 2 Tasks. In the top part of the notebook summarize through a table of content what you decided to do and then explain why.

**Author**: Crystal Zhao    
**Date**: 9/13.    

I decided to attempt Route A, which deals with combining paginated aggregation pages and dedicated pages. My research question will be focused on the top rated movies (5000 results): What is the national composition of top rated movies?   

This would require me to look at this aggregated page: https://mydramalist.com/movies/top. Upon inspection, I would be interested in this element:

"""
<span class="text-muted">Korean Movie - 2017</span>
"""

Rationale/pseudocode: I would go through all pages of this aggregated webpage (250 pages...), and for each page, save the 1) name, 2) rank, and 3) nationality of each movie. For nationality, I would split up the String by space and only extract the first word.

**Table of Contents**

1. [Identifying the Drama Cards](#sec1)
2. [Simple use of `seleniumbase`](#sec2)
3. [Clicking button with `seleniumbase`](#sec3)


<a id="sec1"></a>

## 1. Identifying the Movie Cards

Each movie card is a .box element,   
Each .box contains a .row element, with:  

1. ".ranking.pull-right"
2. ".text-primary-title" class with a hyperlink that directly leads to a dedicated page through <a href> (we should also save this so we can navigate to dedicated pages when necessary)
3. followed by the actual title
3. and finally ".text-muted" (e.g., Chinese Movie - 2019).

In [5]:
from seleniumbase import Driver
from selenium.webdriver.common.by import By
import re
import requests

url = "https://mydramalist.com/movies/top?page=2"

In [1]:
def fetch_page_content(url):
    """Fetches HTML content from a URL and checks status code."""
    response = requests.get(url) # call the function get with the URL, response is a Python object with a lot of attribues

    print(f"URL: {url}")
    print(f"Status Code: {response.status_code}")

    if response.status_code == 200: # check the status code to make sure we got the page from the server 
        return response.text # text is an attribute 
    else:
        print(f"Failed to fetch page. Status code: {response.status_code}")
        return None

In [6]:
page = fetch_page_content(url)

URL: https://mydramalist.com/movies/top?page=2
Status Code: 200


Now that we have the static page html, we can try to parse it with Selenium first to make sure we are looking in the right places for the information we want.

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(page, "html.parser")
top_movies = soup.find('div', class_='col-lg-8 col-md-8') # have to first narrow down to this so we ar enot including other boxes 
movies = top_movies.find_all("div", class_="box")
print(len(movies))

20


The tricky part for me right now is getting both the title and the link, so I am gonna investigate.

In [31]:
movie_1 = movies[0]
tags = movie_1.find_all('a')
print(len(tags)) 
print(tags)
tags[0]

3
[<a class="block" href="/24153-be-with-you">
<img alt="Be with You" class="img-responsive cover lazy" data-src="https://i.mydramalist.com/pnvlns.jpg?v=1"/>
</a>, <a href="/24153-be-with-you">Be with You</a>, <a class="btn simple btn-manage-list" data-id="24153" data-stats="mylist:24153" rel="nofollow"><span><i class="far fa-plus"></i></span></a>]


<a class="block" href="/24153-be-with-you">
<img alt="Be with You" class="img-responsive cover lazy" data-src="https://i.mydramalist.com/pnvlns.jpg?v=1"/>
</a>

What we need is the first a tag that is a block with the link, so we only need to do _find but let's verify that:

In [32]:
title_el = movie_1.find('a')
print(title_el)

<a class="block" href="/24153-be-with-you">
<img alt="Be with You" class="img-responsive cover lazy" data-src="https://i.mydramalist.com/pnvlns.jpg?v=1"/>
</a>


In [33]:
title_el.text

'\n\n'

We can't just call text, so we need to get the alternative description under image.

In [36]:
img_tag = movie_1.find('img')
img_tag['alt']

'Be with You'

In [ ]:
movie_data = []
for m in movies:
        movie_dict = {} 
        
        movie_dict['rank'] = m.find(class_='ranking pull-right').get_text()

        title_tag = movie_1.find('img')
        movie_dict['title'] = title_tag['alt']

        link_el = m.find('a')
        movie_dict['link'] = link_el['href']
        
        if movie_dict:
            movie_data.append(movie_dict)

In [38]:
movie_data[0]

{'rank': '#21', 'title': 'Be with You', 'link': '/24153-be-with-you'}

Success! Now we should be able to wrap this in a function but in Selenium format.

<a id="sec2"></a>

## 2. Defining the 'parse_movies' function

We would call this function after we get to each page, so we would run this function for each page of information that we get.

With this in mind, let's try to scrape just the first page (all movies) and save the information in 1 list.

In [ ]:
def parse_movie(boxes):
    """
    Given a batch of cards, extracts information from the entire drama.
    """
    if not boxes:
        return []

    page_movies_data = []
    
    for movie in boxes:
        movie_dict = {}

        # drama_dict['rank'] = card.find_element(By.CSS_SELECTOR, '.ranking pull-right').text # i'm having trouble with this
        # Compound class names are not allowed.: drama_dict['rank'] = card.find_element(By.CLASS_NAME, '.ranking pull-right').text
        drama_dict['rank'] = card.find_element(By.CSS_SELECTOR, '.ranking.pull-right').text 
        drama_dict['title'] = card.find_element(By.CSS_SELECTOR, '.title').text
        
        meta = card.find_element(By.CSS_SELECTOR, '.text-muted').text
        meta_list = re.split(r'[-,]',meta)
        cleaned_list = [item.strip() for item in meta_list if item.strip()]
        drama_dict['type'] = cleaned_list[0]
        drama_dict['year'] = cleaned_list[1]
        drama_dict['eps'] = cleaned_list[2]

        others = card.find_elements(By.TAG_NAME, 'p') # wrapped in paragraph tags


        drama_dict['rating'] = others[0].text
        drama_dict['synopsis'] = others[1].text

        if drama_dict:
            page_cards_data.append(drama_dict)

    return page_cards_data

In [ ]:
with Driver() as driver:
    driver.open(url)

    cards = driver.find_elements('.box')

    page_1 = []

    for card in cards:
        movie_dict = {}

        movie_dict['rank'] = card.find_element(By.CSS_SELECTOR, '.ranking.pull-right').text 
        # movie_dict['title'] = card.find_element(By.CSS_SELECTOR, '.text-primary-title').text
        title = card.find_element(By.TAG_NAME, "a")
        movie_dict['title'] = title.text
        movie_dict['link'] = title.get_attribute('href')

        meta = card.find_element(By.CSS_SELECTOR, '.text-muted').text
        meta_list = re.split(r'[-,]',meta)
        cleaned_list = [item.strip() for item in meta_list if item.strip()]
        movie_dict['nationality'] = cleaned_list[0]

        if movie_dict:
            page_1.append(movie_dict)
    
page_1[:2]

KeyboardInterrupt: 

I'm running into a lot of trouble with how long it takes the code to run... I also got a ReadTimeoutError: HTTPConnectionPool(host='localhost', port=52864): Read timed out. (read timeout=120) error.

In [ ]:
import math
from seleniumbase import Driver

url = "https://mydramalist.com/movies/top"

with Driver() as driver:
    driver.open(url)

    # 1. Extract total results count (e.g., "5000 results" -> 5000)
    total_text = driver.get_text("p.pull-right") # This will be a string value
    total_results = int(total_text.split()[0])

    # 2. Calculate total pages (20 items per page)
    total_pages = math.ceil(total_results / 20)
    print(f"Total results: {total_results} | Total pages: {total_pages}")

    # 3. Loop through each page URL, but since this is just for demonstration, I will make the end limit 5
    for page in range(1, 5):
        driver.open(f"{url}?page={page}")
        driver.sleep(1.0)

        # 4. Extract items on the current page
        cards = driver.find_elements(".box") # this is a list of cards as WebElements # the . means that the class is box; with a # means ID, with no dot or hash means HTML tag  
        print(f"Page {page}: Scraped {len(cards)} dramas")

<a id="sec2"></a>

## 2. Getting All Movie Cards & Relevant Information

<a id="sec3"></a>

## 3. Going to Dedicated Pages